In [90]:
import numpy as np
import pandas as pd
import sklearn
import joblib
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.metrics import f1_score, precision_score, accuracy_score, recall_score
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\panlw\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\panlw\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\panlw\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [91]:
# load data
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')
STOPWORDS = list(stopwords.words("english"))

In [92]:
train_df.shape

(1500, 13)

In [93]:
X_train, y_train = train_df["Combined_Review"], train_df["source"]
X_test,  y_test  = test_df["Combined_Review"],  test_df["source"]
sentiment_test = test_df["Sentiment"]  #for q3 only
cv = StratifiedKFold(n_splits= 5, shuffle= True, random_state= 42)

In [94]:
X_train.shape

(1500,)

In [95]:
# BoW testing
vectorizer = CountVectorizer()
bows = vectorizer.fit_transform(X_train, y_train)

feature_names = vectorizer.get_feature_names_out()
print(feature_names)


['00' '000' '01' ... 'zero' 'zone' 'élysées']


In [96]:
# Multinomial naive Bayes
multi_nb = sklearn.naive_bayes.MultinomialNB()
multi_nb.fit(bows, y_train)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None
Name,Type,Value
"class_count_ class_count_: ndarray of shape (n_classes,)Number of samples encountered for each class during fitting. Thisvalue is weighted by the sample weight when provided.","ndarray[float64](2,)","[750.,750.]"
"class_log_prior_ class_log_prior_: ndarray of shape (n_classes,)Smoothed empirical log probability for each class.","ndarray[float64](2,)","[-0.69,-0.69]"
"classes_ classes_: ndarray of shape (n_classes,)Class labels known to the classifier","ndarray[int64](2,)","[0,1]"
"feature_count_ feature_count_: ndarray of shape (n_classes, n_features)Number of samples encountered for each (class, feature)during fitting. This value is weighted by the sample weight whenprovided.","ndarray[float64](2, 4769)","[[4.,1.,1.,...,2.,1.,0.], [0.,0.,0.,...,0.,0.,3.]]"
"feature_log_prob_ feature_log_prob_: ndarray of shape (n_classes, n_features)Empirical log probability of featuresgiven a class, ``P(x_i|y)``.","ndarray[float64](2, 4769)","[[ -8.97, -9.89, -9.89,..., -9.48, -9.89,-10.58], [-10.71,-10.71,-10.71,...,-10.71,-10.71, -9.33]]"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,4769


In [97]:
multi_nb_pipeline = Pipeline(
    [('vectorizer', CountVectorizer()),
     ('top_k', SelectKBest(chi2)),
    ('mnb', sklearn.naive_bayes.MultinomialNB())
    ]
)

multi_nb_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('vectorizer', ...), ('top_k', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True


In [98]:
cross_val_score(multi_nb_pipeline, X_train, y_train, cv = 5)

array([0.72333333, 0.71666667, 0.69666667, 0.69      , 0.7       ])

In [99]:
multi_nb_pipeline[:-1].get_feature_names_out()

array(['and', 'attractions', 'no', 'quite', 'rooms', 'service', 'the',
       'was', 'we', 'were'], dtype=object)

In [100]:

param_grid = {
    "vectorizer__min_df": [1, 2, 5],
    "top_k__k": [50, 100, 250, 500, 1000, 2000, "all"],
    "mnb__alpha": [0.01, 0.1, 0.5, 1.0],
    "vectorizer__stop_words": [None, STOPWORDS]
}


grid_search = GridSearchCV(multi_nb_pipeline, param_grid, cv = cv, scoring = 'accuracy', n_jobs= -1)

grid_search.fit(X_train, y_train)
print(grid_search.best_params_)
print(grid_search.best_score_)  



{'mnb__alpha': 0.5, 'top_k__k': 2000, 'vectorizer__min_df': 2, 'vectorizer__stop_words': ['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 'can', 'couldn', "couldn't", 'd', 'did', 'didn', "didn't", 'do', 'does', 'doesn', "doesn't", 'doing', 'don', "don't", 'down', 'during', 'each', 'few', 'for', 'from', 'further', 'had', 'hadn', "hadn't", 'has', 'hasn', "hasn't", 'have', 'haven', "haven't", 'having', 'he', "he'd", "he'll", 'her', 'here', 'hers', 'herself', "he's", 'him', 'himself', 'his', 'how', 'i', "i'd", 'if', "i'll", "i'm", 'in', 'into', 'is', 'isn', "isn't", 'it', "it'd", "it'll", "it's", 'its', 'itself', "i've", 'just', 'll', 'm', 'ma', 'me', 'mightn', "mightn't", 'more', 'most', 'mustn', "mustn't", 'my', 'myself', 'needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on', 'once', 'only', 'or', 'other', 'o

In [101]:
grid_search.best_params_


{'mnb__alpha': 0.5,
 'top_k__k': 2000,
 'vectorizer__min_df': 2,
 'vectorizer__stop_words': ['a',
  'about',
  'above',
  'after',
  'again',
  'against',
  'ain',
  'all',
  'am',
  'an',
  'and',
  'any',
  'are',
  'aren',
  "aren't",
  'as',
  'at',
  'be',
  'because',
  'been',
  'before',
  'being',
  'below',
  'between',
  'both',
  'but',
  'by',
  'can',
  'couldn',
  "couldn't",
  'd',
  'did',
  'didn',
  "didn't",
  'do',
  'does',
  'doesn',
  "doesn't",
  'doing',
  'don',
  "don't",
  'down',
  'during',
  'each',
  'few',
  'for',
  'from',
  'further',
  'had',
  'hadn',
  "hadn't",
  'has',
  'hasn',
  "hasn't",
  'have',
  'haven',
  "haven't",
  'having',
  'he',
  "he'd",
  "he'll",
  'her',
  'here',
  'hers',
  'herself',
  "he's",
  'him',
  'himself',
  'his',
  'how',
  'i',
  "i'd",
  'if',
  "i'll",
  "i'm",
  'in',
  'into',
  'is',
  'isn',
  "isn't",
  'it',
  "it'd",
  "it'll",
  "it's",
  'its',
  'itself',
  "i've",
  'just',
  'll',
  'm',
  'ma',
 

In [102]:
best_mnb = grid_search.best_estimator_
# or this : final_mnb = mnb_pipeline.set_params(**grid_search.best_params_) to use the best parsms

In [104]:
y_pred_mnb = best_mnb.predict(X_test)
f1_mnb = f1_score(y_test, y_pred_mnb)
acc_mnb = accuracy_score(y_test, y_pred_mnb)
recall_mnb = recall_score(y_test, y_pred_mnb)
pre_mnb= precision_score(y_test, y_pred_mnb)
print('MNB result')
print('f1',f1_mnb)
print('-----')
print('accuracy: ', acc_mnb)
print('-----')
print('recall:', recall_mnb)
print('-----')
print('precision: ', pre_mnb)
print('-----')

# save y_pred_mnb for stat test
y_pred_mnb_df = pd.DataFrame(y_pred_mnb)
y_pred_mnb_df.to_csv("data/y_pred_mnb.csv", index=False)

MNB result
f1 0.9024856596558317
-----
accuracy:  0.898
-----
recall: 0.944
-----
precision:  0.8644688644688645
-----


In [ ]:
grid_search.cv_results_

{'mean_fit_time': array([0.06568551, 0.06797242, 0.06083241, 0.05210814, 0.04526892,
        0.04981108, 0.06416459, 0.0629921 , 0.06432872, 0.06077938,
        0.05657754, 0.05871558, 0.07997999, 0.07640376, 0.07633147,
        0.070995  , 0.07490387, 0.05881639, 0.08809171, 0.08841825,
        0.06724858, 0.06401353, 0.06106043, 0.06931992, 0.08256841,
        0.07766719, 0.07972364, 0.06142235, 0.07319074, 0.07141237,
        0.07345557, 0.0811975 , 0.06241479, 0.07380252, 0.06810408,
        0.0638464 , 0.08678765, 0.08412018, 0.08949513, 0.06812363,
        0.07180815, 0.06688604, 0.09748759, 0.10031638, 0.07867117,
        0.08197865, 0.0801898 , 0.07455626, 0.08837724, 0.08138113,
        0.07152033, 0.06637292, 0.07582526, 0.05430527, 0.08330207,
        0.08974056, 0.07011518, 0.06348143, 0.06460514, 0.06306787,
        0.08933959, 0.0802    , 0.08686004, 0.07877741, 0.07835207,
        0.08076553, 0.09315205, 0.09686656, 0.06641617, 0.06023374,
        0.06483207, 0.06262288,

In [ ]:
# save best grid search
res = pd.DataFrame(grid_search.cv_results_)
res["param_vectorizer__stop_words"] = res["param_vectorizer__stop_words"].map(
    lambda s: "none" if s is None else "nltk")
res = res.drop(columns="params")   # also contains the full list
res.to_csv("models/mnb_cv_results.csv", index=False)

In [ ]:
best_mnb[:-1].get_feature_names_out()   # 2k selected words




array(['00', '10', '100', ..., 'yuan', 'zero', 'élysées'],
      shape=(2000,), dtype=object)

In [ ]:
joblib.dump(best_mnb, 'models/mnb.joblib')


['models/mnb.joblib']

In [ ]:
'the' in best_mnb.named_steps['vectorizer'].vocabulary_

False

In [ ]:
#load model
best_mnb = joblib.load('models/mnb.joblib')

In [ ]:

mnb   = best_mnb.named_steps['mnb']
names = best_mnb[:-1].get_feature_names_out()

log_ratio = mnb.feature_log_prob_[1] - mnb.feature_log_prob_[0]   # log P(w|1) / P(w|0)
order = log_ratio.argsort()

top_class1 = names[order[::-1][:5]]  
top_class0 = names[order[:5]]  

In [ ]:
print(top_class0)
print('----')
print(top_class1)

['told' 'booked' 'said' 'com' 'want']
----
['desired' 'offers' 'inconsistent' 'notch' 'underwhelming']
